In [ ]:
import os
from dotenv import load_dotenv
import hopsworks
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

load_dotenv()

project = hopsworks.login(
    api_key_value=os.getenv("HOPSWORKS_API_KEY"),
    project=os.getenv("HOPSWORKS_PROJECT_NAME"),
)
fs = project.get_feature_store()
fg = fs.get_feature_group(name="aqi_features", version=1)
df = fg.read()

df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Total rows: {len(df)}")
df.head()

In [ ]:
df[["target_aqi", "pm25", "pm10", "temperature", "humidity"]].describe()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["timestamp"], df["target_aqi"], color="darkred")
plt.title("AQI Trend Over Time — Karachi")
plt.xlabel("Date")
plt.ylabel("AQI")
plt.axhline(y=150, color="orange", linestyle="--", label="Unhealthy threshold (150)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df["month_name"] = df["timestamp"].dt.month_name()
monthly_avg = df.groupby("month_name")["target_aqi"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=monthly_avg.index, y=monthly_avg.values, palette="Reds_r")
plt.title("Average AQI by Month")
plt.ylabel("Average AQI")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_names = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
df["day_name"] = df["day_of_week"].map(day_names)
weekly_avg = df.groupby("day_name")["target_aqi"].mean().reindex(day_names.values())

plt.figure(figsize=(8, 5))
sns.barplot(x=weekly_avg.index, y=weekly_avg.values, palette="Blues_d")
plt.title("Average AQI by Day of Week")
plt.ylabel("Average AQI")
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ["target_aqi", "pm25", "pm10", "no2", "so2", "co", "o3", "temperature", "humidity"]
corr_df = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap — AQI vs Pollutants & Weather")
plt.tight_layout()
plt.show()

In [ ]:
def categorize(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Moderate"
    elif aqi <= 150: return "Unhealthy (Sensitive)"
    elif aqi <= 200: return "Unhealthy"
    elif aqi <= 300: return "Very Unhealthy"
    else: return "Hazardous"

df["aqi_category"] = df["target_aqi"].apply(categorize)
category_counts = df["aqi_category"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(category_counts.values, labels=category_counts.index, autopct="%1.1f%%",
        colors=sns.color_palette("YlOrRd", len(category_counts)))
plt.title("Distribution of AQI Categories — Karachi")
plt.tight_layout()
plt.show()

print(category_counts)